<a href="https://colab.research.google.com/github/MichaelZimm20/xn-sitstayforever-semantic-attention/blob/main/notebooks/xn_sitstayforever_models_and_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# ------ IMPORTS & SETUP ------
'''
Grad-CAM implementation adapted from:
- Selvaraju et al. (2017), "Grad-CAM: Visual Explanations from Deep Networks
  via Gradient-based Localization" (original Grad-CAM formulation)
- Michael's own prior implementation, AAI6640 Applied Deep Learning final
  project (Brain Tumor MRI CNN classification), Spring 2026


This section adapts the GradCAM concepts to run against the CLIP's visual encoder rather than a standard
image classifer.
- uses cosine similarity to target keyword as the backward pass signal instead of classification logit
- implementing a wrapper using the cosine similarity score to compute the CLIP image/text embedding back to the GradCAM hooks

Purpose:
IS to generate an attention heatmap showing where the models attentions lies when evalauting an image against key words. Theese
keywords are common search terms and listing words that people use for their products
- Example " groomer approved dry shampoo"
This wll help connect the keyword-alignment tool in Notebook 1 to GradCAM
'''


# IMPORTS
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image


# IMPORT CLIP
# Needed for RN50, it has conv layers that GradCAM utilizes
# Installs for missing dependencies not natively supported
!pip install open-clip-torch -q

print("Open CLIP install successful!")
print('~'*50)
# check cpu versus gpu on device using torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


# DRIVE SETUP AND STRUCTURE
BASE_DIR = '/content/drive/MyDrive/xn-sitstayforever'
DATASETS_DIR = f'{BASE_DIR}/datasets'
OUTPUTS_DIR = f'{BASE_DIR}/outputs'
CHECKPOINTS_DIR = f'{BASE_DIR}/checkpoints'

# lOAD NOTEBOOK 1 CSVs outputs
  # training set
clip_train = pd.read_csv(f'{OUTPUTS_DIR}/clip_train.csv')
  # validation set
clip_ssf_eval = pd.read_csv(f'{OUTPUTS_DIR}/clip_ssf_eval.csv')
  # merge train and eval clip scores
clip_scores_merged = pd.concat([clip_train, clip_ssf_eval], ignore_index=True)

# reload notebook 1, 20 keywords used for CLIP and keyword alignment
text_queries = clip_train['all_target_keywords'].iloc[0].split(',')


print('=' * 50)
print('Imports and Setup are successful!')
print('=' * 50)
print(f'Loaded {len(clip_scores_merged)} CLIP images total')



Open CLIP install successful!
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Using device: cpu
Imports and Setup are successful!
Loaded 54 CLIP images total


In [13]:
# force a fresh remount, in case this session's mount is stale
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# then re-check
asin_prefix = 'B0002DINX6'
matches = [f for f in os.listdir(DATASETS_DIR) if f.startswith(asin_prefix)]
print(f"Files starting with '{asin_prefix}': {matches if matches else '(none found)'}")

Mounted at /content/drive
Files starting with 'B0002DINX6': (none found)


In [14]:
'''Smoke Test for verfiying directories, paths, and datasets are loading properly'''
import os

# File path smoke test, since files are in a google drive and mounted
# for item in os.listdir(BASE_DIR):
#   print(repr(item))
print('=' * 50)
for item in os.listdir(DATASETS_DIR):
  print(item)
print('=' * 50)
# check exact folder name including hidden characters
items = os.listdir(BASE_DIR)
for item in items:
  print(repr(item))

# check it path exists
print(os.path.exists(DATASETS_DIR))

SSF_CV_Dataset.xlsx
pet_cv_dataset_full.xlsx
product_images
'datasets'
'checkpoints'
'outputs'
True


In [4]:
# ----- LOAD CLIP for RN50 ------
'''
Notebook 1 used ViT-B-32 for CLIP Scoring = a zero shot checkpoint
- GradCAM utilizes convolutional feature maps, ViT doesnt require this,
a secondary seperate CLIP checkpoint with RN50 used specifically for GradCAM


reference - Ilharco et al. (2021), OpenCLIP (open-source implementation used here,
  https://github.com/mlfoundations/open_clip)


'''

# IMPORT OPEN CLIP
import open_clip

clip_model, _, clip_preprocessing = open_clip.create_model_and_transforms(
    model_name='RN50',
    pretrained='openai',
)

clip_tokenizer = open_clip.get_tokenizer('RN50')

# Use GPU is present, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
clip_model.to(device)
clip_model.eval()

print('/n/nModel configuration and loading successful!')
print('=' * 50)
print(f'CLIP model: RN-50 (pretrained)')
print(f'Device: {device}')
print(f'Visual type {type(clip_model.visual)}')
print(f'Text type {type(clip_model.token_embedding)}')

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


/n/nModel configuration and loading successful!
CLIP model: RN-50 (pretrained)
Device: cpu
Visual type <class 'open_clip.modified_resnet.ModifiedResNet'>
Text type <class 'torch.nn.modules.sparse.Embedding'>


In [31]:
'''
------- CLIPGradCAM class implementation -------

It makes more sense to put all of CLIP/GradCAM methods into one major class
- this limits us having a bunch of loose functions and trying to remember each one to call. Can simply return to the class


References:
- https://docs.pytorch.org/docs/2.13/generated/torch.nn.modules.module.register_module_forward_hook.html


'''

class CLIPGradCAM:
  def __init__(self, model, tokenizer, preprocess, device, target_layer=None):
    self.model = model
    self.tokenizer = tokenizer
    self.preprocess = preprocess
    self.device = device


  # Forward and Backward Hook, happens only when the class is created and not on generation of the heat map\
    # default target layer
    self.target_layer = target_layer if target_layer is not None else self.model.visual.layer4[-1]
      # gradients and activations
    self.gradients = None
    self.activations = None


    # register forward and backward hook
    self.target_layer.register_forward_hook(self._forward_hook)
    self.target_layer.register_backward_hook(self._backward_hook)

  # define the hook functions
  def _forward_hook(self, module, input, output):
    self.activations = output

  def _backward_hook(self, module, grad_in, grad_out):
    self.gradients = grad_out[0].detach() # break the tensor

  # load and preprocess the images from the datasets
  def load_and_preprocess_image(self, image_path):
    # load image from disk, then apply CLIP RN50 preprocessing
    image = self.preprocess(Image.open(image_path).convert('RGB')).unsqueeze(0).to(self.device)
    return image

  # get text embeddings
  def get_text_embeddings(self, keyword):
    # disable gradient calculations, save memory, boost speed, backprogagation not need
    with torch.no_grad():
      # tokenize keywords to 1D tensor of integers
       keyword_to_token = self.tokenizer([keyword]).to(self.device)

       # from raw integet tokens to embeddings (pushed through the models NN - Transformer)
       text_features = self.model.encode_text(keyword_to_token)
       text_features = text_features / text_features.norm(dim=-1, keepdim=True) # L2 normalization
    return text_features

  # implement the same functionality to get image embeddings
  def get_image_embeddings(self, input_tensor):
    '''
      - encode a preproccessed image tensor into a normalized CLIP image embedding
    '''
    # load image from disk, then apply CLIP RN50 preprocessing

    # from raw integet tokens to embeddings (pushed through the models NN - Transformer)
    image_features = self.model.encode_image(input_tensor)
    image_features = image_features / image_features.norm(dim=-1, keepdim=True) # L2 normalization
    return image_features # shape: [1, embed_dim]

  # get cosine similarity scores between image and text embedding
  def cosine_sim_scores(self, image_embedding, text_embedding):
    '''
      - Combining cosine Similarity scores for image and text embeddings
      - putting this in a the class, and reworking the same code from notebook 1 for reuseability

    '''
    return (image_embedding @ text_embedding.T).squeeze() #squeeze and tranpose( flipping the matrix, and remove unneccessary dimensions)


  # generate the full CLIP GradCAM pipeline for the image/keyword pair
  def generate_clip_gradcam(self, image_path, keyword):
    self.model.zero_grad() # zeroing the gradients

    # input_tensor, image_embedding, text_embedding
    input_tensor = self.load_and_preprocess_image(image_path)
    text_embedding = self.get_text_embeddings(keyword)
    image_embedding = self.get_image_embeddings(input_tensor)


    # compute cosine similarity score
    similarity_score = self.cosine_sim_scores(image_embedding, text_embedding)

    # apply backward pass, using the chain rule to calculate rate of change from gradients
    similarity_score.backward()

    # add weights to activations, ReLU activation function, then normalize
     # zero out the bad noise and keep the good and target
    weights = self.gradients.mean(dim=[2, 3], keepdim=True)
    g_cam = (weights * self.activations).sum(dim=1, keepdim=True) #retain original number of dimensions of the tensor
    g_cam = torch.relu(g_cam)
    g_cam = g_cam.squeeze().detach().cpu().numpy() # squeeze down to a 2D array

    #
    if g_cam.max() > 0:
      g_cam = g_cam / g_cam.max()

    return g_cam, similarity_score.item()

# instantiate - hooks and add to same class
clip_gradcam = CLIPGradCAM(clip_model, clip_tokenizer, clip_preprocessing, device)
print('CLIPGradCAM class has been initialized !!')
print('G_cam generator and hooks are added !!')


CLIPGradCAM class has been initialized !!
G_cam generator and hooks are added !!


In [19]:
# ---- Smoke Test: sanity check to test if CLIPGradCAM class is working -----------
sample_row = clip_scores_merged.iloc[0]
sample_path = os.path.join(DATASETS_DIR, 'product_images', sample_row['image_filename'])

sample_image_tensor = clip_gradcam.load_and_preprocess_image(sample_path)
sample_text_embedding = clip_gradcam.get_text_embeddings(text_queries[0])
sample_image_embedding = clip_gradcam.get_image_embeddings(sample_image_tensor)

sample_score = clip_gradcam.cosine_sim_scores(sample_image_embedding, sample_text_embedding)
print(f'Image: {sample_row['image_filename']}')
print(f'Keyword: {text_queries[0]}')
print(f'Similarity Score: {sample_score.item():.4f}')


Image: B0002DINX6_img01.jpg
Keyword: labrador odor control powder
Similarity Score: 0.1563


In [32]:
# ----- 2nd Smoke Test ---------
'''
- running second smoke test with cam generator and hooks added.

will include cam shape, similiarity score, and value range of GradCAM


ran into a "numpy on a tensor that requires grad error. Utilize detach() to resolve it

https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html
'''
sample_row = clip_scores_merged.iloc[0]
sample_path = os.path.join(DATASETS_DIR, 'product_images', sample_row['image_filename'])

cam, score = clip_gradcam.generate_clip_gradcam(sample_path, text_queries[0]) # unpack the scores tuple
print(f'Image: {sample_row['image_filename']}')
print(f'Keyword: {text_queries[0]}')
print(f'Similarity Score: {score:.4f}')
print(f'GradCAM shape: {cam.shape}')
print(f'GradCAM value range: {cam.min():.4f} to {cam.max():.4f}') # get min and max values


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1870: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)


Image: B0002DINX6_img01.jpg
Keyword: labrador odor control powder
Similarity Score: 0.1563
GradCAM shape: (7, 7)
GradCAM value range: 0.0000 to 1.0000
